# 09 Dash Visualizations

Interactive dashboard for sessions and ML outputs using DuckDB-backed datasets.

In [ ]:
from pathlib import Path

import plotly.express as px
from dash import Dash, Input, Output, dcc, html

from utils.duckdb_utils import query_df, table_exists
from utils.paths import get_db_paths, resolve_workspace

workspace = resolve_workspace(Path.cwd())
input_db, output_db = get_db_paths(workspace)

sessions = query_df(input_db, 'SELECT title, day, track, conference, level_name, status FROM sessions_in_preprocessed')
dr_df = None
if output_db.exists() and table_exists(output_db, 'dr_embeddings'):
    dr_df = query_df(output_db, 'SELECT title, day, track, conference, svd_1, svd_2 FROM dr_embeddings')
            


In [ ]:
app = Dash(__name__)
days = sorted(sessions['day'].dropna().unique().tolist())
tracks = sorted(sessions['track'].dropna().unique().tolist())

app.layout = html.Div([
    html.H2('FabCon Sessions + ML Explorer'),
    html.Div([
        dcc.Dropdown(days, value=days[0] if days else None, id='day-dd', placeholder='Day'),
        dcc.Dropdown(tracks, value=tracks[0] if tracks else None, id='track-dd', placeholder='Track'),
    ], style={'display': 'grid', 'gridTemplateColumns': '1fr 1fr', 'gap': '12px'}),
    dcc.Graph(id='bar-fig'),
    dcc.Graph(id='sun-fig'),
    dcc.Graph(id='emb-fig')
])

@app.callback(
    Output('bar-fig', 'figure'),
    Output('sun-fig', 'figure'),
    Output('emb-fig', 'figure'),
    Input('day-dd', 'value'),
    Input('track-dd', 'value')
)
def update(day_value, track_value):
    filt = sessions.copy()
    if day_value:
        filt = filt[filt['day'] == day_value]
    if track_value:
        filt = filt[filt['track'] == track_value]

    counts = filt.groupby(['conference', 'level_name'], dropna=False).size().reset_index(name='n')
    fig_bar = px.bar(counts, x='level_name', y='n', color='conference', barmode='group', title='Sessions by Level and Conference')

    sb = filt.groupby(['conference', 'day', 'status'], dropna=False).size().reset_index(name='n')
    fig_sun = px.sunburst(sb, path=['conference', 'day', 'status'], values='n', title='Status Composition')

    if dr_df is not None:
        emb = dr_df.copy()
        if day_value:
            emb = emb[emb['day'] == day_value]
        if track_value:
            emb = emb[emb['track'] == track_value]
        fig_emb = px.scatter(emb, x='svd_1', y='svd_2', color='conference', hover_name='title', title='SVD Embedding Scatter')
    else:
        fig_emb = px.scatter(title='Run notebook 05 to materialize dr_embeddings.')

    return fig_bar, fig_sun, fig_emb

app

In [ ]:
# Inline mode if supported by your Dash version; otherwise use app.run(debug=False).
app.run(jupyter_mode='inline', debug=False, port=8050)